In [48]:
from langchain_openai import ChatOpenAI 
from langchain_groq import ChatGroq
from langchain.document_loaders import  PyPDFLoader
from langchain.vectorstores import  FAISS
from langchain.text_splitter import  RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings 
from langchain.prompts import PromptTemplate
from langchain.docstore.document import Document
from langchain.chains.summarize import load_summarize_chain
from langchain_core.pydantic_v1 import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.runnables.graph import MermaidDrawMethod

from langgraph.graph import END, StateGraph

from time import monotonic
from dotenv import load_dotenv
from pprint import pprint
import os
from datasets import Dataset
from typing_extensions import TypedDict
from IPython.display import display, Image
from typing import TypedDict, Literal, Optional, List

from ragas import evaluate
from ragas.metrics import (
    answer_correctness,
    faithfulness,
    answer_relevancy,
    context_recall,
    answer_similarity
)

import langgraph


load_dotenv(override=True)

os.environ["PYDEVD_WARN_EVALUATION_TIMEOUT"] = "100000"

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()  # 读取 .env

# 用 DeepSeek 的 key 伪装成 OPENAI_API_KEY，让 ChatOpenAI 觉得自己有 key
os.environ["OPENAI_API_KEY"] = os.getenv("DEEPSEEK_API_KEY")
os.environ["OPENAI_API_BASE"] = "https://api.deepseek.com"


In [26]:
IntentType = Literal["faq", "tech_issue", "account", "other"]

class SupportState(TypedDict, total=False):
    """TechSupport-Agent 在 graph 里的共享状态."""
    query: str                           # 当前用户问题
    intent: IntentType                   # 意图分类结果
    answer: str                          # 当前节点生成的回答（最后可以用于输出）
    retrieved_docs: List[Document]       # 如果有检索，就放这里（目前只有 FAQ 用）

### 数据集构建（简易版）

In [4]:
pdf_path ="deepseek.pdf"

loader = PyPDFLoader(pdf_path)
raw_docs = loader.load()


len(raw_docs), raw_docs[0][:200] if isinstance(raw_docs[0], str) else raw_docs[0]

(2,
 Document(page_content='快速开始 首次调用 API\n首次调用 API\nDeepSeek API 使用与 OpenAI 兼容的 API 格式，通过修改配置，您可以使用 OpenAI SDK 来访\n问 DeepSeek API，或使用与 OpenAI API 兼容的软件。\nPARAM VALUE\nbase_url *       https://api.deepseek.com\napi_k ey apply for an API k ey\n* 出于与 OpenAI 兼容考虑，您也可以将 base_url 设置为 https://api.deepseek.com/v1 来使用，但注\n意，此处 v1 与模型版本无关。\n* deepseek-chat 和 deepseek-reasoner 都已经升级为 DeepSeek -V3.2。deepseek-chat 对应 DeepSeek -\nV3.2 的 非思考模式 ，deepseek-reasoner 对应 DeepSeek -V3.2 的 思考模式 。\n调用对话 API\n在创建 API k ey 之后，你可以使用以下样例脚本的来访问 DeepSeek API。样例为非流式输出，\n您可以将 str eam 设置为 true 来使用流式输出。\ncurl py thon nodejs\ncurl https://api.deepseek.com/chat/completions \\\n  -H "Content-Type: application/json" \\\n  -H "Authorization: Bearer ${DEEPSEEK_API_KEY}" \\\n  -d \'{\n        "model": "deepseek-chat",\n        "messages": [\n          {"role": "system", "content": "You are a helpful assistant."},\n          {"role": "user", "content": "Hello!"}\n        ],\n        "stream": false\n      }\'2025/12/8 19:02 ⾸

### 分块

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,     # 每块大概 800 字符
    chunk_overlap=100,  # 相邻块有一点重叠，避免句子被硬拆开
)

docs = text_splitter.split_documents(raw_docs)

len(docs), docs[0]

(3,
 Document(page_content='快速开始 首次调用 API\n首次调用 API\nDeepSeek API 使用与 OpenAI 兼容的 API 格式，通过修改配置，您可以使用 OpenAI SDK 来访\n问 DeepSeek API，或使用与 OpenAI API 兼容的软件。\nPARAM VALUE\nbase_url *       https://api.deepseek.com\napi_k ey apply for an API k ey\n* 出于与 OpenAI 兼容考虑，您也可以将 base_url 设置为 https://api.deepseek.com/v1 来使用，但注\n意，此处 v1 与模型版本无关。\n* deepseek-chat 和 deepseek-reasoner 都已经升级为 DeepSeek -V3.2。deepseek-chat 对应 DeepSeek -\nV3.2 的 非思考模式 ，deepseek-reasoner 对应 DeepSeek -V3.2 的 思考模式 。\n调用对话 API\n在创建 API k ey 之后，你可以使用以下样例脚本的来访问 DeepSeek API。样例为非流式输出，\n您可以将 str eam 设置为 true 来使用流式输出。\ncurl py thon nodejs\ncurl https://api.deepseek.com/chat/completions \\\n  -H "Content-Type: application/json" \\\n  -H "Authorization: Bearer ${DEEPSEEK_API_KEY}" \\\n  -d \'{\n        "model": "deepseek-chat",\n        "messages": [', metadata={'source': 'deepseek.pdf', 'page': 0}))

### 向量化

In [ ]:
def replace_t_with_space(list_of_documents):
    for doc in list_of_documents:
        doc.page_content = doc.page_content.replace('\t', ' ')  # Replace tabs with spaces
    return list_of_documents

def encode_book(path, chunk_size=1000, chunk_overlap=200):
    """
    Encodes a PDF book into a vector store using HuggingFace embeddings.
    """
    loader = PyPDFLoader(path)
    documents = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap, length_function=len
    )
    texts = text_splitter.split_documents(documents)
    cleaned_texts = replace_t_with_space(texts)

    embeddings = HuggingFaceEmbeddings(
        model_name="BAAI/bge-small-en-v1.5",
        encode_kwargs={"normalize_embeddings": True},
    )
    vectorstore = FAISS.from_documents(cleaned_texts, embeddings)

    return vectorstore


In [7]:
faq_vectorstore = encode_book(pdf_path)
faq_retriever = faq_vectorstore.as_retriever(search_kwargs={"k": 4})

faq_vectorstore, faq_retriever

(<langchain_community.vectorstores.faiss.FAISS at 0x76f2b8e62d20>,
 VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x76f2b8e62d20>, search_kwargs={'k': 4}))

### 构建IntentClassifier链

In [27]:
class IntentResult(BaseModel):
    intent: IntentType = Field(
        description="用户问题的意图，必须是 'faq', 'tech_issue', 'account', 或 'other' 之一。"
    )
    reason: str = Field(
        description="用简短中文解释为什么做出这个意图判断。"
    )

In [29]:
intent_prompt_template="""
你是一个技术支持平台的意图分类助手，需要把用户的问题归类到以下四类之一：

1. faq：纯文档/说明类问题，常见模式：
   - “这个接口怎么用？”
   - “某个参数是什么意思？”
   - “错误码 401 的含义是什么？”
   重点在于查文档就能回答的说明类问题。

2. tech_issue：环境 / 报错 / 排错类问题，常见模式：
   - “我调用接口报 401/429/5xx 怎么办？”
   - “向量数据库连接失败 connection refused？”
   - “docker compose 启动某个服务失败？”
   重点在于“出错了，需要一步步排查”。

3. account：账号 / 计费 / 配额 / 限速咨询，常见模式：
   - “免费额度用完会发生什么？”
   - “为什么提示 quota exceeded？”
   - “这个 key 有没有开通某个权限？”
   重点在于账号状态、套餐、配额、计费策略等。

4. other：不符合以上三类的其他问题，或者难以判断的情况。

【用户问题】
{query}

请根据问题内容，给出 intent（faq / tech_issue / account / other）以及简短的 reason。
""".strip()

intent_prompt = PromptTemplate(
    template=intent_prompt_template,
    input_variables=["query"],
)

intent_llm = ChatOpenAI(
    temperature=0, model_name="deepseek-chat", max_tokens=512,
)

intent_chain = intent_prompt | intent_llm.with_structured_output(IntentResult)

In [30]:
def intent_classifier_node(state: SupportState) -> SupportState:
    """根据 state['query'] 判断意图，并写回 state['intent']。"""
    query = state["query"]
    result: IntentResult = intent_chain.invoke({"query": query})
    state["intent"] = result.intent
    # 如果你后面想在调试时看看 reason，也可以临时 print 一下：
    # print("[Intent]", result.intent, "| reason:", result.reason)
    return state

#### test

In [31]:
tests = [
    "如何调用 DeepSeek 的 chat 接口？请求体需要哪些字段？",
    "我调用 API 一直报 401 unauthorized，怎么办？",
    "免费额度用完之后接口还能用吗？",
    "你觉得大模型会统治世界吗？",
]

for q in tests:
    r = intent_chain.invoke({"query": q})
    print("Q:", q)
    print("intent:", r.intent)
    print("reason:", r.reason)
    print("-" * 60)


Q: 如何调用 DeepSeek 的 chat 接口？请求体需要哪些字段？
intent: faq
reason: 用户询问如何调用DeepSeek的chat接口以及请求体需要哪些字段，这属于API使用方法和参数说明的文档类问题，可以通过查阅API文档来回答。
------------------------------------------------------------
Q: 我调用 API 一直报 401 unauthorized，怎么办？
intent: tech_issue
reason: 用户描述调用API报401 unauthorized错误，属于环境/报错/排错类问题，需要一步步排查认证失败的原因
------------------------------------------------------------
Q: 免费额度用完之后接口还能用吗？
intent: account
reason: 用户询问免费额度用完后接口是否还能使用，这涉及到账号的计费策略、配额限制和套餐状态，属于账号相关的咨询问题。
------------------------------------------------------------
Q: 你觉得大模型会统治世界吗？
intent: other
reason: 用户询问的是关于大模型未来发展的哲学性问题，与技术文档、故障排查、账号计费等技术支持范畴无关，属于其他类型问题。
------------------------------------------------------------


### 构建FAQ链

In [12]:
class FAQAnswer(BaseModel):
    """FAQ 最终答案的结构化输出。"""
    short_answer: str = Field(
        description="用 1-3 句话直接回答用户问题的核心结论。"
    )
    details: str = Field(
        description="更详细的说明，可以包含参数解释、使用建议等。"
    )
    #caveats: Optional[str] = Field(
    #    default=None,
    #    description="如果文档中有未说明的点，或者需要提示用户注意的地方，在这里说明。",
    #)


In [16]:
faq_answer_prompt_template = """
你是一名熟悉 DeepSeek API 文档的技术支持工程师。

【用户问题】
{question}

【与问题相关的文档内容】（已经经过预处理，只保留了和问题强相关的部分）
{relevant_content}

请严格基于上述文档内容回答用户的问题，不要编造文档中没有的信息。
回答时遵循以下要求：

1. short_answer：用 1-3 句中文直接回答用户问题的核心结论。
2. details：用一段或几段话，详细说明相关接口/参数/错误码的含义和使用方式，可以使用列表或分点说明。
3. caveats：如果文档中没有包含用户想问的某些信息，或者有需要特别提醒用户注意的地方（例如限速、权限、配额等），在这里简要说明；如果没有，可以写“无”或留空。

请按上述字段生成结构化输出。
""".strip()

faq_answer_prompt = PromptTemplate(
    template=faq_answer_prompt_template,
    input_variables=["question", "relevant_content"],
)

faq_answer_llm = ChatOpenAI(
    temperature=0, model_name="deepseek-chat", max_tokens=2000
)

faq_answer_chain = faq_answer_prompt | faq_answer_llm.with_structured_output(FAQAnswer)

In [22]:
def answer_faq(question: str, top_k: int = 4) -> FAQAnswer:
    """
    FAQ v1（structured_output 版）：
    1. 用 faq_retriever 检索 top_k 段文本
    2. 拼成 context
    3. 调用 faq_structured_chain，返回 FAQAnswer 对象
    """
    # 1. 检索知文档
    docs: List[Document] = faq_retriever.get_relevant_documents(question)[:top_k]

    # 2. 粗暴拼接上下文（先不做“只保留相关内容”的中间层）
    context = "\n\n---\n\n".join(
        f"[page={d.metadata.get('page', '')}]\n{d.page_content}"
        for d in docs
    )

    # 3. 调用 structured_output 链
    result: FAQAnswer = faq_answer_chain.invoke(
        {"question": question, "relevant_content": context}
    )

    return result

In [32]:
def faq_node(state: SupportState) -> SupportState:
    question = state["query"]          # 这里把 state 的 query 映射给 answer_faq 的 question
    faq_result = answer_faq(question)  # 调用你上面的链

    final_answer = (
        f"【FAQ 简要回答】\n{faq_result.short_answer}\n\n"
        f"【FAQ 详细说明】\n{faq_result.details}"
    )
    if faq_result.caveats and faq_result.caveats.strip() and faq_result.caveats.strip() != "无":
        final_answer += f"\n\n【注意事项】\n{faq_result.caveats}"

    state["answer"] = final_answer
    return state

#### test

In [33]:
q1 = "如何调用 DeepSeek 的 chat 接口？请求体需要哪些字段？"
q2 = "temperature 参数是干什么用的？一般怎么设置？"

for q in [q1, q2]:
    print("=" * 80)
    print("问题：", q)
    ans = answer_faq(q)
    print("\n[short_answer]")
    print(ans.short_answer)
    print("\n[details]")
    print(ans.details)
    print("\n\n")


问题： 如何调用 DeepSeek 的 chat 接口？请求体需要哪些字段？



[short_answer]
调用 DeepSeek 的 chat 接口需要使用 POST 请求到 https://api.deepseek.com/chat/completions，请求体必须包含 model、messages 和 stream 字段。

[details]
根据文档内容，调用 DeepSeek chat 接口的具体步骤如下：

1. **API 端点**：POST 请求到 `https://api.deepseek.com/chat/completions`

2. **请求头**：
   - `Content-Type: application/json`
   - `Authorization: Bearer ${DEEPSEEK_API_KEY}`（需要替换为实际的 API Key）

3. **请求体必需字段**：
   - `model`：指定使用的模型，如 `"deepseek-chat"`（对应 DeepSeek-V3.2 的非思考模式）
   - `messages`：对话消息数组，包含角色和内容，如：
     ```json
     [
       {"role": "system", "content": "You are a helpful assistant."},
       {"role": "user", "content": "Hello!"}
     ]
     ```
   - `stream`：是否使用流式输出，设置为 `false` 为非流式输出，设置为 `true` 为流式输出

4. **模型说明**：
   - `deepseek-chat`：对应 DeepSeek-V3.2 的非思考模式
   - `deepseek-reasoner`：对应 DeepSeek-V3.2 的思考模式（文档中提到但未在示例中使用）

5. **兼容性**：DeepSeek API 使用与 OpenAI 兼容的 API 格式，可以通过修改配置使用 OpenAI SDK 访问。



问题： temperature 参数是干什么用的？一般怎么设置？

[short_answer]
根据提供的文档内容，文档中没有明确提到 temperature 参数的具体作用和设置方法。

[details]
在提供的 Dee

### 构建techissue链

In [34]:
class TechIssueAnswer(BaseModel):
    """技术问题（报错/环境）的结构化排错输出。"""
    summary: str = Field(
        description="用 1-3 句中文概括问题本质和大致方向，例如：'这是一个认证失败相关的问题'"
    )
    possible_causes: List[str] = Field(
        description="可能的原因列表，每条是一句话或一小段，例如配置错误、权限不足、网络问题等"
    )
    steps: List[str] = Field(
        description="推荐的排查步骤列表，按顺序执行（Step 1, Step 2...），每条一步"
    )

In [35]:
tech_issue_prompt_template = """
你是一名熟悉 DeepSeek API 文档的技术支持工程师，专门帮助用户排查“调用出错 / 环境问题”。

【用户问题】
{question}

【与问题相关的文档内容】（来自官方文档，可能包含错误码说明、限速说明、接口用法示例等）
{relevant_content}

请严格基于上述文档内容进行分析，不要编造文档中没有的信息。
请按以下结构化方式输出排查建议（对应 TechIssueAnswer 模型）：

1. summary：
   - 用 1-3 句中文概括这个问题大概是哪一类（例如认证失败、配额耗尽、限速、请求格式错误等）。
   - 如果文档中没有足够信息确定具体原因，请使用“可能是……，需要进一步确认”的语气。

2. possible_causes：
   - 输出一个列表，每一项是一个“可能的原因”，例如：
     - API Key 无效或没有对应权限
     - 请求中 model 字段填写错误
     - 触发了限速或配额限制
   - 这些原因必须能够从文档内容推断出来，或者是文档中明确提到的常见场景。

3. steps：
   - 输出一个“按顺序排查的步骤”列表，例如：
     - 第一步：在控制台确认 API Key 是否有效
     - 第二步：确认请求头 Authorization 是否正确设置
     - 第三步：检查请求体中 model / messages / content 是否符合文档要求
   - 每条步骤尽量具体，可直接给用户操作建议。

如果文档中完全没有提到与该问题相关的信息，请在 summary 中说明这一点，
并在 possible_causes 和 steps 中给出通用的、保守的建议（例如检查网络、检查 key、查看控制台日志等）。
""".strip()

tech_issue_prompt = PromptTemplate(
    template=tech_issue_prompt_template,
    input_variables=["question", "relevant_content"],
)

tech_issue_llm = ChatOpenAI(
    model="deepseek-chat",
    temperature=0,
    max_tokens=2000,
)

tech_issue_chain = tech_issue_prompt | tech_issue_llm.with_structured_output(TechIssueAnswer)

In [36]:
def answer_tech_issue(question: str, top_k: int = 4) -> TechIssueAnswer:
    """
    Tech Issue v1（structured_output 版）：
    1. 用（暂时复用的）faq_retriever 检索 top_k 段文本
    2. 拼成 relevant_content
    3. 调用 tech_issue_chain，返回 TechIssueAnswer 对象
    """
    docs: List[Document] = faq_retriever.get_relevant_documents(question)[:top_k]

    relevant_content = "\n\n---\n\n".join(
        f"[page={d.metadata.get('page', '')}]\n{d.page_content}"
        for d in docs
    )

    result: TechIssueAnswer = tech_issue_chain.invoke(
        {
            "question": question,
            "relevant_content": relevant_content,
        }
    )

    return result

In [43]:
def tech_issue_node(state: SupportState) -> SupportState:
    """Tech issue 处理节点：用 TechIssueAnswer 结构输出排错建议。"""
    question = state["query"]
    issue_result = answer_tech_issue(question)

    # 格式化为最终展示文本（以后可以在 UI 里直接用结构化字段）
    parts = [
        f"【问题概述】\n{issue_result.summary}",
        "\n【可能原因】",
    ]
    for i, cause in enumerate(issue_result.possible_causes, start=1):
        parts.append(f"{i}. {cause}")
    parts.append("\n【建议排查步骤】")
    for i, step in enumerate(issue_result.steps, start=1):
        parts.append(f"{i}. {step}")

    state["answer"] = "\n".join(parts)
    return state

### account链

In [40]:
class AccountAnswer(BaseModel):
    """
    账号 / 计费 / 配额 类问题的结构化输出。
    """
    short_answer: str = Field(
        description="用 1-3 句中文，直接说明用户问题的核心结论。"
    )
    details: str = Field(
        description="更详细的说明，包括计费规则、配额/限速机制、常见情形等。"
    )
    next_steps: Optional[str] = Field(
        default=None,
        description="给用户的后续操作建议，例如去控制台哪里查看用量、如何更换套餐、如何联系客服等。"
    )


In [41]:
account_answer_prompt_template = """
你是一名熟悉 DeepSeek 平台计费与配额规则的客服工程师。

【用户问题】
{question}

【与问题相关的文档内容】（来自官方文档，可能包含计费说明、配额规则、限速策略等）
{relevant_content}

请严格基于上述文档内容回答用户的问题，不要编造文档中没有的信息。
回答时遵循以下要求（对应 AccountAnswer 模型的字段）：

1. short_answer：
   - 用 1-3 句中文，直接说明用户问题的核心结论。
   - 例如：说明“免费额度用完后，接口会返回配额耗尽错误；需要升级套餐或等待重置”。

2. details：
   - 更详细地解释相关规则，可以包括：
     - 计费方式（按调用量、按 token、按模型等）
     - 免费额度 / 试用额度的限制
     - 配额耗尽或限速时的典型错误码与表现
   - 可以使用列表或分点说明，但仍然要基于文档内容。

3. next_steps：
   - 给出用户可以执行的后续操作建议，例如：
     - 在控制台某个页面查看用量和账单
     - 确认当前 API Key 是否有对应权限
     - 考虑升级套餐、开通付费、或者联系人工客服
   - 如果文档中没有提供明确建议，可以给出通用的、保守的建议；如果实在没法建议，可以写“无”。

注意：
- 你无法访问用户的真实账号、用量或订单信息，只能给出通用说明和建议。
- 如果文档中没有包含用户关心的某个细节，请在 details 中说明“文档中未提到这一点”，不要编造具体数值或政策。
""".strip()

account_answer_prompt = PromptTemplate(
    template=account_answer_prompt_template,
    input_variables=["question", "relevant_content"],
)

account_answer_llm = ChatOpenAI(
    model="deepseek-chat",
    temperature=0,
    max_tokens=2000,
)

account_answer_chain = account_answer_prompt | account_answer_llm.with_structured_output(AccountAnswer)

In [42]:
def answer_account(question: str, top_k: int = 4) -> AccountAnswer:
    """
    Account v1（structured_output 版）：
    1. 用（暂时复用的）faq_retriever 检索 top_k 段文本
    2. 拼成 relevant_content
    3. 调用 account_answer_chain，返回 AccountAnswer 对象
    """
    docs: List[Document] = faq_retriever.get_relevant_documents(question)[:top_k]

    relevant_content = "\n\n---\n\n".join(
        f"[page={d.metadata.get('page', '')}]\n{d.page_content}"
        for d in docs
    )

    result: AccountAnswer = account_answer_chain.invoke(
        {
            "question": question,
            "relevant_content": relevant_content,
        }
    )

    return result

In [44]:
def account_node(state: SupportState) -> SupportState:
    """Account 处理节点：调用 answer_account，并把结果写回 state。"""
    question = state["query"]
    acc_result = answer_account(question)

    parts = [
        f"【账号/计费简要说明】\n{acc_result.short_answer}",
        f"\n【详细说明】\n{acc_result.details}",
    ]
    if acc_result.next_steps and acc_result.next_steps.strip() and acc_result.next_steps.strip() != "无":
        parts.append(f"\n【后续建议】\n{acc_result.next_steps}")

    state["answer"] = "\n".join(parts)
    # 如果你也想保存这次检索到的 docs，可以改 answer_account 返回 (result, docs)
    # 然后这里赋值 state["retrieved_docs"] = docs
    return state

#### test

In [45]:
q1 = "免费额度用完之后，接口会发生什么？"
q2 = "为什么会报 quota exceeded 这种错误？一般怎么处理？"

for q in [q1, q2]:
    print("=" * 80)
    print("问题：", q)
    acc = answer_account(q)
    print("\n[short_answer]")
    print(acc.short_answer)
    print("\n[details]")
    print(acc.details)
    print("\n[next_steps]")
    print(acc.next_steps)
    print("\n\n")

问题： 免费额度用完之后，接口会发生什么？

[short_answer]
根据提供的文档内容，文档中没有明确说明免费额度用完之后接口会发生什么。文档主要介绍了API的首次调用和基本使用方法。

[details]
文档中包含了以下相关信息：
1. API调用格式：展示了如何使用curl命令调用DeepSeek API
2. 模型说明：提到deepseek-chat对应DeepSeek-V3.2的非思考模式，deepseek-reasoner对应思考模式
3. API兼容性：说明DeepSeek API使用与OpenAI兼容的API格式

然而，文档中完全没有提及：
- 计费方式和定价策略
- 免费额度的具体限制
- 配额耗尽时的处理机制
- 错误码和限速策略
- 免费额度用完后接口的具体行为

文档内容主要集中在技术实现层面，没有包含账号管理、计费规则或配额限制的相关信息。

[next_steps]
由于文档中没有提供相关信息，建议：
1. 访问DeepSeek官方控制台查看账户用量和计费信息
2. 查阅完整的官方文档，特别是计费相关章节
3. 联系DeepSeek客服获取准确的配额和计费政策说明
4. 在控制台中检查API Key的状态和剩余额度



问题： 为什么会报 quota exceeded 这种错误？一般怎么处理？

[short_answer]
根据文档内容，quota exceeded 错误通常表示 API 调用配额已耗尽，可能是免费额度用完或套餐限制导致的。

[details]
基于提供的文档内容，我注意到以下几点：

1. **文档中明确提到的信息**：
   - DeepSeek API 需要申请 API Key 才能使用
   - API 调用格式与 OpenAI 兼容
   - 支持 deepseek-chat（非思考模式）和 deepseek-reasoner（思考模式）两种模型

2. **文档中未明确提到的信息**：
   - 具体的计费方式和价格策略
   - 免费额度或试用额度的具体数值
   - 配额耗尽时的具体错误码和表现细节
   - 配额重置周期或升级方式

3. **可能的配额相关情况**：
   - 新用户可能有初始免费额度，用完后会出现配额错误
   - 不同套餐可能有不同的调用限制
   - 可能存在按时间

In [49]:
def other_node(state: SupportState) -> SupportState:
    """兜底节点：当意图不是 faq / tech_issue / account 时的回复。"""
    query = state["query"]
    state["answer"] = (
        "目前这个 TechSupport-Agent 主要支持以下几类问题：\n"
        "1) API / 参数 / 错误码等文档类问题（faq）\n"
        "2) 报错/环境/排错类问题（tech_issue）\n"
        "3) 账号 / 计费 / 配额类问题（account）\n\n"
        f"你刚才的问题暂时被归类为 'other'：\n\n"
        f"「{query}」\n\n"
        "你可以尝试：\n"
        "- 更具体地描述你调用的接口、报错信息\n"
        "- 说明你关注的是：用法？报错？还是配额/计费？"
    )
    return state

### graph骨架

##### 条件边

In [ ]:
def route_after_intent(state: SupportState) -> str:
    """
    根据 state['intent'] 决定下一步走哪个节点。
    返回值要跟 add_conditional_edges 里的 key 对得上。
    """
    intent = state.get("intent", "other")
    if intent not in ("faq", "tech_issue", "account", "other"):
        return "other"
    return intent

In [51]:
# 1. 定义工作流，状态类型用 SupportState
workflow = StateGraph(SupportState)

# 2. 注册各个节点函数（你前面已经实现好了）
workflow.add_node("intent_classifier", intent_classifier_node)
workflow.add_node("faq", faq_node)
workflow.add_node("tech_issue", tech_issue_node)
workflow.add_node("account", account_node)
workflow.add_node("other", other_node)

# 3. 从 START 进图，先跑意图分类
workflow.set_entry_point("intent_classifier")

# 4. 条件路由函数
def route_after_intent(state: SupportState) -> str:
    intent = state.get("intent", "other")
    if intent not in ("faq", "tech_issue", "account", "other"):
        return "other"
    return intent

# 5. 从 intent_classifier 出发，根据 intent 去不同节点
workflow.add_conditional_edges(
    "intent_classifier",
    route_after_intent,
    {
        "faq": "faq",
        "tech_issue": "tech_issue",
        "account": "account",
        "other": "other",
    },
)


# 6. 告诉 LangGraph：这几个节点走完就是结束（连到 END）
workflow.add_edge("faq", END)
workflow.add_edge("tech_issue", END)
workflow.add_edge("account", END)
workflow.add_edge("other", END)

support_agent_app = workflow.compile()

### test

In [52]:
q = "我调用 DeepSeek 的 chat 接口总是 401 unauthorized，怎么排查？"
result = support_agent_app.invoke({"query": q})
print("intent:", result.get("intent"))
print("answer:\n", result.get("answer", ""))


intent: tech_issue
answer:
 【问题概述】
这是一个API认证失败的问题，具体表现为401 unauthorized错误，通常与API Key配置或请求头设置有关。

【可能原因】
1. API Key无效或未正确申请
2. Authorization请求头格式错误
3. API Key权限不足或已过期

【建议排查步骤】
1. 第一步：确认已申请有效的API Key，文档中提到需要'apply for an API key'
2. 第二步：检查请求头Authorization格式是否正确，文档示例为'Authorization: Bearer ${DEEPSEEK_API_KEY}'，确保Bearer后有空格且key正确
3. 第三步：验证base_url设置，文档显示应为https://api.deepseek.com或https://api.deepseek.com/v1
4. 第四步：检查请求体格式，确保model字段为'depseek-chat'或'depseek-reasoner'，messages格式符合文档示例
